# Agente Inteligente para Clasificación de Potabilidad del Agua

Este notebook usa un **CSV online** de potabilidad del agua y construye un agente inteligente que:

1. Carga los datos.
2. Prepara las variables.
3. Entrena varios algoritmos de clasificación.
4. Evalúa cada modelo con **AUC**.
5. Selecciona automáticamente el mejor modelo.
6. Interpreta los resultados de forma automática.


In [ ]:
# ==========================================
# LIBRERÍAS
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.metrics import roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    BaggingClassifier
)

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

import warnings
warnings.filterwarnings("ignore")


In [ ]:
# ==========================================
# CLASE DEL AGENTE INTELIGENTE
# ==========================================

class AgenteInteligentePotabilidad:

    def __init__(self, archivo_csv, variable_objetivo="Potability"):
        self.archivo_csv = archivo_csv
        self.variable_objetivo = variable_objetivo
        self.df = None
        self.X = None
        self.y = None
        self.resultados = {}

    def cargar_datos(self):
        self.df = pd.read_csv(self.archivo_csv)

        print("\nDatos cargados correctamente")
        print(self.df.head())

        print("\nTamaño del dataset:")
        print(self.df.shape)

    def preparar_datos(self):
        self.X = self.df.drop(columns=[self.variable_objetivo])
        self.y = self.df[self.variable_objetivo]

        print("\nVariables predictoras:")
        print(self.X.columns.tolist())

        print("\nVariable objetivo:")
        print(self.variable_objetivo)

        print("\nDistribución de clases:")
        print(self.y.value_counts())

    def obtener_modelos(self):
        modelos = {
            "Logistic Regression": LogisticRegression(max_iter=2000),
            "Decision Tree": DecisionTreeClassifier(random_state=42),

            "Random Forest": RandomForestClassifier(
                n_estimators=200,
                random_state=42
            ),

            "Extra Trees": ExtraTreesClassifier(
                n_estimators=200,
                random_state=42
            ),

            "Gradient Boosting": GradientBoostingClassifier(
                random_state=42
            ),

            "AdaBoost": AdaBoostClassifier(
                random_state=42
            ),

            "Bagging": BaggingClassifier(
                random_state=42
            ),

            "SVM": SVC(
                probability=True,
                random_state=42
            ),

            "KNN": KNeighborsClassifier(),

            "Naive Bayes": GaussianNB(),

            "MLP Neural Network": MLPClassifier(
                hidden_layer_sizes=(100, 50),
                max_iter=2000,
                random_state=42
            )
        }

        return modelos

    def entrenar_y_evaluar(self):
        X_train, X_test, y_train, y_test = train_test_split(
            self.X,
            self.y,
            test_size=0.25,
            random_state=42,
            stratify=self.y
        )

        columnas_numericas = self.X.columns.tolist()

        preprocesador = ColumnTransformer(
            transformers=[
                (
                    "num",
                    Pipeline([
                        ("imputer", SimpleImputer(strategy="mean")),
                        ("scaler", StandardScaler())
                    ]),
                    columnas_numericas
                )
            ]
        )

        modelos = self.obtener_modelos()

        print("\n====================================")
        print("ENTRENAMIENTO DE MODELOS")
        print("====================================")

        for nombre, modelo in modelos.items():

            pipeline = Pipeline([
                ("preprocessing", preprocesador),
                ("model", modelo)
            ])

            pipeline.fit(X_train, y_train)

            probabilidades = pipeline.predict_proba(X_test)[:, 1]

            auc = roc_auc_score(y_test, probabilidades)

            self.resultados[nombre] = {
                "modelo": pipeline,
                "auc": auc
            }

            print(f"{nombre:<25} AUC = {auc:.4f}")

    def seleccionar_mejor_modelo(self):
        mejor_modelo = max(
            self.resultados,
            key=lambda x: self.resultados[x]["auc"]
        )

        print("\n====================================")
        print("MEJOR MODELO")
        print("====================================")
        print(f"Modelo: {mejor_modelo}")
        print(f"AUC: {self.resultados[mejor_modelo]['auc']:.4f}")

        return self.resultados[mejor_modelo]["modelo"]

    def interpretar_resultados(self):
        df_resultados = pd.DataFrame([
            {
                "Modelo": modelo,
                "AUC": info["auc"]
            }
            for modelo, info in self.resultados.items()
        ])

        df_resultados = df_resultados.sort_values(
            by="AUC",
            ascending=False
        )

        mejor = df_resultados.iloc[0]
        peor = df_resultados.iloc[-1]
        promedio_auc = df_resultados["AUC"].mean()

        print("\n====================================")
        print("INTERPRETACIÓN AUTOMÁTICA")
        print("====================================")

        print(f"Mejor modelo: {mejor['Modelo']}")
        print(f"AUC del mejor modelo: {mejor['AUC']:.4f}")

        if mejor["AUC"] >= 0.90:
            nivel = "EXCELENTE"
        elif mejor["AUC"] >= 0.80:
            nivel = "MUY BUENO"
        elif mejor["AUC"] >= 0.70:
            nivel = "ACEPTABLE"
        elif mejor["AUC"] >= 0.60:
            nivel = "BAJO"
        else:
            nivel = "DEFICIENTE"

        print(f"Interpretación del desempeño: {nivel}")

        print("\n¿Qué significa el AUC?")
        print(
            "El AUC mide la capacidad del modelo para distinguir "
            "entre agua potable y agua no potable. "
            "Mientras más cercano a 1, mejor es el modelo."
        )

        print("\nComparación general:")
        print(f"AUC promedio de todos los modelos: {promedio_auc:.4f}")
        print(f"Peor modelo: {peor['Modelo']}")
        print(f"AUC del peor modelo: {peor['AUC']:.4f}")

        diferencia = mejor["AUC"] - promedio_auc

        print("\nConclusión del agente:")

        if diferencia > 0.05:
            print(
                f"El modelo {mejor['Modelo']} supera claramente "
                "el rendimiento promedio de los demás algoritmos."
            )
        else:
            print(
                f"El modelo {mejor['Modelo']} fue el mejor, "
                "pero su ventaja respecto al promedio no es muy amplia."
            )

        print(
            "\nRecomendación: usar el mejor modelo como candidato principal "
            "y validarlo posteriormente con validación cruzada."
        )

        return df_resultados

    def ejecutar(self):
        self.cargar_datos()
        self.preparar_datos()
        self.entrenar_y_evaluar()

        mejor_modelo = self.seleccionar_mejor_modelo()

        tabla_resultados = self.interpretar_resultados()

        return self.resultados, mejor_modelo, tabla_resultados


In [ ]:
# ==========================================
# DATASET ONLINE
# ==========================================

url = "https://raw.githubusercontent.com/Sarthak-1408/Water-Potability/main/water_potability.csv"


In [5]:
# ==========================================
# EJECUCIÓN DEL AGENTE
# ==========================================

agente = AgenteInteligentePotabilidad(
    archivo_csv=url,
    variable_objetivo="Potability"
)

resultados, mejor_modelo, tabla_resultados = agente.ejecutar()



Datos cargados correctamente
         ph    Hardness        Solids  Chloramines     Sulfate  Conductivity  \
0       NaN  204.890455  20791.318981     7.300212  368.516441    564.308654   
1  3.716080  129.422921  18630.057858     6.635246         NaN    592.885359   
2  8.099124  224.236259  19909.541732     9.275884         NaN    418.606213   
3  8.316766  214.373394  22018.417441     8.059332  356.886136    363.266516   
4  9.092223  181.101509  17978.986339     6.546600  310.135738    398.410813   

   Organic_carbon  Trihalomethanes  Turbidity  Potability  
0       10.379783        86.990970   2.963135           0  
1       15.180013        56.329076   4.500656           0  
2       16.868637        66.420093   3.055934           0  
3       18.436524       100.341674   4.628771           0  
4       11.558279        31.997993   4.075075           0  

Tamaño del dataset:
(3276, 10)

Variables predictoras:
['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'O

In [7]:
# ==========================================
# TABLA FINAL ORDENADA POR AUC
# ==========================================

tabla_resultados


,Modelo,AUC
3,Extra Trees,0.679628
2,Random Forest,0.652721
7,SVM,0.645873
10,MLP Neural Network,0.624248
4,Gradient Boosting,0.613837
8,KNN,0.604741
6,Bagging,0.602602
9,Naive Bayes,0.599255
5,AdaBoost,0.578980
1,Decision Tree,0.557139


In [8]:
# ==========================================
# EJEMPLO DE PREDICCIÓN CON EL MEJOR MODELO
# ==========================================

nuevo_dato = pd.DataFrame({
    "ph": [7.2],
    "Hardness": [180],
    "Solids": [22000],
    "Chloramines": [7],
    "Sulfate": [330],
    "Conductivity": [450],
    "Organic_carbon": [14],
    "Trihalomethanes": [78],
    "Turbidity": [4.5]
})

prediccion = mejor_modelo.predict(nuevo_dato)
probabilidad = mejor_modelo.predict_proba(nuevo_dato)

print("Predicción:", prediccion[0])
print("Probabilidad de no potable:", probabilidad[0][0])
print("Probabilidad de potable:", probabilidad[0][1])


Predicción: 0
Probabilidad de no potable: 0.67
Probabilidad de potable: 0.33
